# Calculate Intensity Features

Computes per-modality first-order statistics (mean, variance, skewness, kurtosis) for tumor vs. non-tumor voxels in BraTS 2023. Outputs `GLI-Image_intensity_tumor_vs_non_tumor.pkl`.


In [ ]:
from tqdm import tqdm
import os
from os import listdir
import time
from random import randint
from os.path import isfile, join

from scipy.stats import skew, kurtosis
import pickle as pkl
 
import gc 
import numpy as np
from scipy import stats
import pandas as pd


import nibabel as nib

from skimage import feature

import matplotlib.pyplot as plt
from matplotlib import cm








import warnings
warnings.simplefilter("ignore")

In [ ]:
def preprocess_mask_labels(mask):
    # whole tumour
    mask_WT = mask.copy()
    mask_WT[mask_WT == 1] = 1
    mask_WT[mask_WT == 2] = 1
    mask_WT[mask_WT == 3] = 1
    # include all tumours 

    # NCR / NET - LABEL 1
    mask_TC = mask.copy()
    mask_TC[mask_TC == 1] = 1
    mask_TC[mask_TC == 2] = 0
    mask_TC[mask_TC == 3] = 1
    # exclude 2 / 4 labelled tumour 

    # ET - LABEL 4 
    mask_ET = mask.copy()
    mask_ET[mask_ET == 1] = 0
    mask_ET[mask_ET == 2] = 0
    mask_ET[mask_ET == 3] = 1
    # exclude 2 / 1 labelled tumour 

    mask = np.stack([mask_WT, mask_TC, mask_ET, mask_ET])
    
    return mask 

def get_tumor_slices(mri_mask):
    """
    Extracts slices from a 3D MRI mask where the tumor exists.

    Parameters:
    mri_mask (numpy.ndarray): A 3D NumPy array representing the MRI mask.

    Returns:
    list of numpy.ndarray: A list of 2D slices containing the tumor.
    """
    tumor_slices = []

    # Iterate through each slice
    for i in range(mri_mask.shape[2]):
        _slice = mri_mask[:, :, i]
        
        # Check if the slice contains tumor (non-zero values)
        if np.any(_slice):
            tumor_slices.append(i)

    return tumor_slices

def read_MRI(dataset, patient_id):
    if dataset == 'Brats2020':
        baseloc = '../input/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData/'
        pefix = 'BraTS20_Training_' + patient_id + '/' + 'BraTS20_Training_' + patient_id
        suffixs = ['_flair.nii','_t2.nii', '_t1.nii', '_t1ce.nii', '_seg.nii']
    elif dataset == 'Brats2023':
        baseloc = '../input/Brats2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
        pefix = 'BraTS-GLI-' + patient_id + '/' + 'BraTS-GLI-' + patient_id
        suffixs = ['-t2f.nii.gz','-t2w.nii.gz', '-t1n.nii.gz', '-t1c.nii.gz', '-seg.nii.gz']

    flair_filename = baseloc + pefix + suffixs[0]
    flair_img_f = nib.load(flair_filename)
    flair_img = np.asarray(flair_img_f.dataobj)

    t2_filename = baseloc + pefix + suffixs[1]
    t2_img_f = nib.load(t2_filename)
    t2_img = np.asarray(t2_img_f.dataobj)

    t1_filename = baseloc + pefix + suffixs[2]
    t1_img_f = nib.load(t1_filename)
    t1_img = np.asarray(t1_img_f.dataobj)

    t1ce_filename = baseloc + pefix + suffixs[3]
    t1ce_img_f = nib.load(t1ce_filename)
    t1ce_img = np.asarray(t1ce_img_f.dataobj)
    
    mask_filename = baseloc + pefix + suffixs[4]
    mask_img_f = nib.load(mask_filename)
    mask_img = np.asarray(mask_img_f.dataobj)
    
    return flair_img, t2_img, t1_img, t1ce_img, mask_img 

def calculate_intensity(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    average_intensity_t1 = np.mean(t1_img)
    average_intensity_tice = np.mean(t1ce_img)
    average_intensity_t2 = np.mean(t2_img)
    average_intensity_flair = np.mean(flair_img)
    
    return {'t1':average_intensity_t1, 
            't1ce':average_intensity_tice, 
            't2':average_intensity_t2, 
            'flair':average_intensity_flair}

def calculate_intensity_mask_only(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img   = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
    # Crop to slices containing tumor to reduce noise from empty slices
    t1_img = t1_img[:,:, start_slice:end_slice]
    t1ce_img = t1ce_img[:,:, start_slice:end_slice]
    t2_img = t2_img[:,:, start_slice:end_slice]
    flair_img = flair_img[:,:, start_slice:end_slice]
    
    average_intensity_t1 = np.mean(t1_img)
    average_intensity_tice = np.mean(t1ce_img)
    average_intensity_t2 = np.mean(t2_img)
    average_intensity_flair = np.mean(flair_img)
    
    return {'t1':average_intensity_t1, 
            't1ce':average_intensity_tice, 
            't2':average_intensity_t2, 
            'flair':average_intensity_flair}


def calculate_intensity_tumor(dataset, patient_id):
    flair_img, t2_img, t1_img, t1ce_img, mask_img = read_MRI(dataset, patient_id)
    
    slices = get_tumor_slices(mask_img)
    start_slice = slices[0]
    end_slice = slices[-1]
    
    t1_img = t1_img[:,:, start_slice:end_slice]
    t1ce_img = t1ce_img[:,:, start_slice:end_slice]
    t2_img = t2_img[:,:, start_slice:end_slice]
    flair_img = flair_img[:,:, start_slice:end_slice]
    
    # Cast to bool for boolean indexing of tumor / non-tumor voxels
    mask_img = mask_img[:,:, start_slice:end_slice].astype(bool)

    def calculate_stats(image, mask):
        tumor_values = image[mask]
        non_tumor_values = image[~mask]

        tumor_stats = {
            'average_intensity': np.mean(tumor_values),
            'variance': np.var(tumor_values),
            'skewness': skew(tumor_values),
            'kurtosis': kurtosis(tumor_values)
        }

        non_tumor_stats = {
            'average_intensity': np.mean(non_tumor_values),
            'variance': np.var(non_tumor_values),
            'skewness': skew(non_tumor_values),
            'kurtosis': kurtosis(non_tumor_values)
        }

        return tumor_stats, non_tumor_stats

    t1_stats = calculate_stats(t1_img, mask_img)
    t1ce_stats = calculate_stats(t1ce_img, mask_img)
    t2_stats = calculate_stats(t2_img, mask_img)
    flair_stats = calculate_stats(flair_img, mask_img)

    return {
        'tumor': {
            't1': t1_stats[0],
            't1ce': t1ce_stats[0],
            't2': t2_stats[0],
            'flair': flair_stats[0]
        },
        'non-tumor': {
            't1': t1_stats[1],
            't1ce': t1ce_stats[1],
            't2': t2_stats[1],
            'flair': flair_stats[1]
        }
    }



In [ ]:
dataset = 'Brats2023'
data_path = '../input/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData/'
patient_ids = [f for f in listdir(data_path) if not isfile(join(data_path, f))]

image_intensities = {}
count = 0
for _id in patient_ids:
    print(_id)
    count += 1
    patient_id = _id.split('GLI-')[1]
    image_intensity = calculate_intensity_tumor(dataset, patient_id)
    
    image_intensities[_id] = image_intensity
#     if count == 2:
#         break
# image_intensities_df = pd.DataFrame.from_dict(image_intensities, 
#                                          orient = 'index', 
#                                          columns = ['t1', 
#                                                     't1ce', 
#                                                     't2', 
#                                                     'flair'])

# image_intensities_df.to_csv('../Results/Analysis_Results/intensity/GLI-Image_intensity_whole.csv')
# with open('../Results/Analysis_Results/intensity/GLI-Image_intensity_whole.pkl', 'wb') as handle:
#     pkl.dump(image_intensities, handle, protocol=pkl.HIGHEST_PROTOCOL)

In [ ]:
with open('../Results/Analysis_Results/intensity/GLI-Image_intensity_tumor_vs_non_tumor.pkl', 'wb') as handle:
    pkl.dump(image_intensities, handle, protocol=pkl.HIGHEST_PROTOCOL)